# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library.

### Dataset Source
The dataset source is a Croissant schema available at:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The dataset may contain more than one record set (table) with fields (columns). Below, we enumerate them by their unique `@id` for reference.

In [ ]:
# List all record sets in the dataset by `@id`
print("Available record sets with @id:")
for record_set in dataset.record_sets:
    print(f"@id: {record_set.id}   |  name: {record_set.name}")

# For demonstration, select the main data table record set. If multiple exist, list fields for each.
selected_record_set = None
if dataset.record_sets:
    selected_record_set = dataset.record_sets[0]  # Use the first as an example
    print(f"\nFields in record set '{selected_record_set.name}' (@id: {selected_record_set.id}):")
    for field in selected_record_set.fields:
        print(f"  @id: {field.id}   |  name: {field.name}   |  dataType: {field.data_type}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Reference all items by their `@id`.

In [ ]:
# Extract data from each record set by its @id
record_set_ids = [r.id for r in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded record set '@id': {record_set_id}, number of records: {len(records)}")

# Preview the columns from the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns in DataFrame for record set '@id': {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply some basic processing steps: filtering records by values, normalizing a numeric field, and grouping by a key field. 
All fields are referenced by their `@id` as per the dataset's schema.

In [ ]:
# For analysis, select a numeric field and a group field, referencing by their @id
if main_record_set_id:
    # Find numeric fields in the main record set
    numeric_fields = [field for field in dataset[main_record_set_id].fields if field.data_type in ("Integer", "Float", "Number")]
    group_fields = [field for field in dataset[main_record_set_id].fields if field.data_type == "Text"]

    if numeric_fields:
        numeric_field_id = numeric_fields[0].id
        print(f"Using numeric field '@id': {numeric_field_id} for EDA")
        threshold = 10

        # Filter records where the numeric field is above threshold
        df = dataframes[main_record_set_id]
        # Coerce column to numeric just in case
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Group by a categorical field if available
        if group_fields:
            group_field_id = group_fields[0].id
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"\nGrouped data by '{group_field_id}':")
            print(grouped_df[[numeric_field_id, normalized_col]].head())
    else:
        print("No numeric fields available for EDA in this record set.")

## 5. Visualization
Visualize data distributions or statistical relationships in the dataset.
We use only the fields' `@id`s for labelling.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize the distribution of the numeric field for the main record set
if main_record_set_id and numeric_fields:
    numeric_field_id = numeric_fields[0].id
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # If a group field is available, plot comparisons
    if group_fields:
        group_field_id = group_fields[0].id
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² dataset by loading its Croissant schema, inspecting its structure via `@id`s, extracting and filtering its data, performing numeric normalization and groupings, and visualizing distributions. 

This approach, referencing all entities by `@id`, maximizes interoperability and reproducibility for advanced analytics workflows.